# Pipeline Demo

End-to-end inference: frame → YOLO detection → IntentionLSTM + SpeedLSTM → predicted ego speed.

Run from the project root: `CS7180-Final/`

In [1]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path("..")
sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

PyTorch: 2.11.0+cpu
CUDA available: False


In [2]:
from pipeline.detector import BoundingBoxEngineering
from pipeline.speed_predictor import SpeedPredictor
from pipeline.runner import PipelineRunner

INTENTION_WEIGHTS = str(PROJECT_ROOT / "weights" / "intention_lstm_best.pt")
SPEED_WEIGHTS     = str(PROJECT_ROOT / "weights" / "speed_lstm_best.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

detector        = BoundingBoxEngineering()          # yolo26n.pt by default
speed_predictor = SpeedPredictor(
    intention_weights=INTENTION_WEIGHTS,
    speed_weights=SPEED_WEIGHTS,
    device=DEVICE,
)
runner = PipelineRunner(
    detector=detector,
    speed_predictor=speed_predictor,
    seq_len=15,
)

print("Pipeline loaded.")

Pipeline loaded.


## Option A — Run on a video file

Set `VIDEO_PATH` to any `.mp4` file and run the cell below.

In [ ]:
VIDEO_PATH = "path/to/your/video.mp4"   # <-- change this
MAX_FRAMES_DISPLAY = 6                   # number of sample frames to visualise

cap = cv2.VideoCapture(VIDEO_PATH)
assert cap.isOpened(), f"Could not open: {VIDEO_PATH}"

fps        = cap.get(cv2.CAP_PROP_FPS)
n_frames   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {n_frames} frames @ {fps:.1f} fps")

frames_rgb = []
speeds     = []
runner._buffer.clear()

while True:
    ret, frame_bgr = cap.read()
    if not ret:
        break
    speed = runner.run_frame(frame_bgr)
    speeds.append(speed)
    frames_rgb.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

cap.release()
print(f"Processed {len(speeds)} frames. "
      f"Speed available for {sum(s is not None for s in speeds)} frames.")

In [ ]:
# ── Speed over time plot ───────────────────────────────────────────────────
valid_idx    = [i for i, s in enumerate(speeds) if s is not None]
valid_speeds = [speeds[i] for i in valid_idx]
times        = [i / fps for i in valid_idx]

plt.figure(figsize=(12, 3))
plt.plot(times, valid_speeds, linewidth=2)
plt.axhline(0, color='red', linestyle='--', linewidth=0.8, label='stopped')
plt.xlabel("Time (s)")
plt.ylabel("Predicted ego speed (km/h)")
plt.title("Predicted ego vehicle speed over time")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Sample frames with predicted speed overlaid ────────────────────────────
sample_indices = np.linspace(0, len(valid_idx) - 1, MAX_FRAMES_DISPLAY, dtype=int)
sample_frames  = [valid_idx[i] for i in sample_indices]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, fi in zip(axes.flat, sample_frames):
    ax.imshow(frames_rgb[fi])
    spd = speeds[fi]
    color = "red" if (spd is not None and spd < 5) else "lime"
    label = f"{spd:.1f} km/h" if spd is not None else "buffering…"
    ax.set_title(f"Frame {fi}  |  {label}", color=color, fontsize=11)
    ax.axis("off")

plt.suptitle("Sample frames with predicted ego speed", fontsize=13)
plt.tight_layout()
plt.show()

## Option B — Run on a single image

Useful for quick sanity checks. Note: speed prediction requires 15 buffered frames,
so a single image will return `None` — but you can inspect the YOLO detections.

In [ ]:
IMAGE_PATH = "path/to/your/image.jpg"   # <-- change this

frame_bgr = cv2.imread(IMAGE_PATH)
assert frame_bgr is not None, f"Could not read: {IMAGE_PATH}"
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

# Raw YOLO detections via BoundingBoxEngineering
result = detector.transform([frame_bgr])   # (1, 10, 224, 224, 8)
frame_data = result[0]                     # (10, 224, 224, 8)

H, W = frame_bgr.shape[:2]
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(frame_rgb)

n_detections = 0
for obj in frame_data:
    meta = obj[0, 0, 3:]           # [x, y, w, h, cls] normalised
    x, y, w, h, cls = meta
    if w == 0 and h == 0:
        continue
    n_detections += 1
    label  = "pedestrian" if cls == 1 else "traffic light"
    color  = "lime" if cls == 1 else "yellow"
    x1_px  = (x - w / 2) * W
    y1_px  = (y - h / 2) * H
    w_px   = w * W
    h_px   = h * H
    rect   = patches.Rectangle((x1_px, y1_px), w_px, h_px,
                                linewidth=2, edgecolor=color, facecolor='none')
    ax.add_patch(rect)
    ax.text(x1_px, y1_px - 5, label, color=color, fontsize=9,
            bbox=dict(facecolor='black', alpha=0.5, pad=1))

ax.set_title(f"YOLO26 detections — {n_detections} object(s) found")
ax.axis("off")
plt.tight_layout()
plt.show()

## Option C — Run on PIE test images

Uses frames from the PIE dataset (if you have the images extracted).

In [ ]:
import glob

# Point to a PIE video's extracted frames, e.g. data/pie/images/set01/video_0001/
FRAMES_DIR = "path/to/pie/images/set01/video_0001"   # <-- change this

image_paths = sorted(glob.glob(f"{FRAMES_DIR}/*.png"))
print(f"Found {len(image_paths)} frames")

runner._buffer.clear()
pie_speeds = []
pie_frames = []

for path in image_paths:
    frame_bgr = cv2.imread(path)
    if frame_bgr is None:
        continue
    speed = runner.run_frame(frame_bgr)
    pie_speeds.append(speed)
    pie_frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))

valid_pie = [(i, s) for i, s in enumerate(pie_speeds) if s is not None]
print(f"Speed predictions for {len(valid_pie)}/{len(pie_speeds)} frames")

if valid_pie:
    idxs, spds = zip(*valid_pie)
    plt.figure(figsize=(12, 3))
    plt.plot(idxs, spds, linewidth=2)
    plt.axhline(0, color='red', linestyle='--', linewidth=0.8)
    plt.xlabel("Frame")
    plt.ylabel("Predicted speed (km/h)")
    plt.title("Predicted ego speed — PIE sequence")
    plt.tight_layout()
    plt.show()